# 1단계: PyTorch 설치 확인 + GPU 동작 확인 + 텐서 기초

**이 노트북에서 배우는 것**

1. PyTorch가 GPU(CUDA)를 인식하는지 확인하는 방법
2. 텐서(Tensor)의 생성과 기본 연산
3. 텐서를 CPU ↔ GPU 사이에서 옮기는 방법 (`.to("cuda")`)
4. 같은 연산이 CPU와 GPU에서 얼마나 속도 차이가 나는지 체감

각 셀을 위에서부터 순서대로 실행하세요 (`Shift + Enter`).

## 1. 환경 확인

`torch.cuda.is_available()`이 `True`이면 GPU 버전 PyTorch가 정상 설치된 것입니다.

In [1]:
import time
import torch

print(f"PyTorch 버전     : {torch.__version__}")
print(f"CUDA 사용 가능?  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU 이름         : {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM 총량        : {vram:.1f} GB")
    device = torch.device("cuda")
else:
    print("GPU를 찾지 못했습니다. CPU로 진행합니다.")
    print("(GPU 버전 PyTorch를 설치했는지, 드라이버가 정상인지 확인하세요)")
    device = torch.device("cpu")

PyTorch 버전     : 2.12.1+cu126
CUDA 사용 가능?  : True
GPU 이름         : NVIDIA GeForce RTX 2060
VRAM 총량        : 6.0 GB


## 2. 텐서 기초

텐서 = PyTorch의 기본 자료구조. NumPy 배열과 비슷하지만 **GPU에 올릴 수 있고, 자동 미분(gradient)을 지원**합니다.

In [2]:
a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])  # 직접 값으로 생성
b = torch.rand(2, 2)                         # 0~1 균등분포 난수로 생성

print(f"a =\n{a}")
print(f"b =\n{b}")
print(f"a + b =\n{a + b}")          # 원소별 덧셈
print(f"a @ b =\n{a @ b}")          # 행렬 곱 (@ 연산자)
print(f"a의 shape: {a.shape}, dtype: {a.dtype}, 위치: {a.device}")

a =
tensor([[1., 2.],
        [3., 4.]])
b =
tensor([[0.5660, 0.9050],
        [0.2586, 0.6468]])
a + b =
tensor([[1.5660, 2.9050],
        [3.2586, 4.6468]])
a @ b =
tensor([[1.0833, 2.1985],
        [2.7326, 5.3019]])
a의 shape: torch.Size([2, 2]), dtype: torch.float32, 위치: cpu


## 3. 텐서를 GPU로 옮기기

딥러닝 코드의 핵심 패턴: **모델과 데이터를 같은 device에 두어야** 연산이 가능합니다. (하나는 CPU, 하나는 GPU면 에러 발생)

In [3]:
a_gpu = a.to(device)                # GPU로 복사 (device가 cuda일 때)
print(f"이동 후 위치: {a_gpu.device}")

c = a_gpu * 10                      # GPU 위에서 연산 -> 결과도 GPU에 있음
print(f"GPU 연산 결과 c의 위치: {c.device}")

c_cpu = c.cpu()                     # 다시 CPU로 (NumPy 변환 등에 필요)
print(f"CPU로 복귀: {c_cpu.device}")

이동 후 위치: cuda:0
GPU 연산 결과 c의 위치: cuda:0
CPU로 복귀: cpu


## 4. CPU vs GPU 속도 비교

큰 행렬 곱셈(4096×4096, 10회)으로 차이를 체감해봅니다.

> 참고: 작은 연산은 GPU 전송 비용 때문에 오히려 CPU가 빠를 수 있습니다!

In [4]:
size = 4096
x_cpu = torch.rand(size, size)
y_cpu = torch.rand(size, size)

# --- CPU 측정 ---
start = time.perf_counter()
for _ in range(10):
    _ = x_cpu @ y_cpu
cpu_time = time.perf_counter() - start
print(f"CPU: {cpu_time:.2f}초")

# --- GPU 측정 ---
if device.type == "cuda":
    x_gpu = x_cpu.to(device)
    y_gpu = y_cpu.to(device)

    _ = x_gpu @ y_gpu               # 워밍업 (첫 호출은 초기화 비용 포함)
    torch.cuda.synchronize()        # GPU 연산은 비동기라서 동기화 후 측정

    start = time.perf_counter()
    for _ in range(10):
        _ = x_gpu @ y_gpu
    torch.cuda.synchronize()        # 모든 GPU 연산이 끝날 때까지 대기
    gpu_time = time.perf_counter() - start
    print(f"GPU: {gpu_time:.2f}초")
    print(f"-> GPU가 약 {cpu_time / gpu_time:.0f}배 빠름")

CPU: 3.56초
GPU: 0.27초
-> GPU가 약 13배 빠름


---
**1단계 완료!** 다음은 `step2_mnist_cnn.ipynb`를 열어보세요.